In [1]:
import sys
from pathlib import Path
# sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
sys.path.insert(0, "../..")

from duckify_simulation.duckify_sim import DuckifySim as ISCoin
# from URBasic import ISCoin  # <-- swap this line to use the real robot

# from src.safety import get_reachable
from URBasic import Joint6D, TCP6D
from URBasic.waypoint6d import TCP6DDescriptor
from math import radians, pi

# Connect
iscoin = ISCoin() #(host="10.30.5.159")

DuckifySim connected to container 'iscoin_simulator'


In [2]:
# Define pen position
HANDE_OFFSET = 0.1005
PEN_OFFSET = 0.128
PEN_IN_SUPPORT_OFFSET = 0.101
SUPPORT_OFFSET = 0.046
DOWN_RX, DOWN_RY, DOWN_RZ = pi, 0.0, 0.0

PEN_POS = [0.3, -0.3, SUPPORT_OFFSET + PEN_IN_SUPPORT_OFFSET]  # pen tip position in world frame -(SUPPORT_OFFSET - PEN_IN_SUPPORT_OFFSET)

# Use TCP offset instead of manually adding HANDE_OFFSET to target Z
tool_tcp = TCP6D.createFromMetersRadians(0.0, 0.0, HANDE_OFFSET, 0.0, 0.0, 0.0)
iscoin.robot_control.set_tcp(tool_tcp)

In [3]:
# Get pen position with Arm TCP
# Use with the real robot arm
# iscoin.robot_control.freedrive_mode()

# i = input()

# pen_tcp = iscoin.robot_control.get_actual_tcp_pose()

# PEN_POS = [pen_tcp.x, pen_tcp.y, pen_tcp.z]

# iscoin.robot_control.end_freedrive_mode()

In [4]:
# 1. Read current position
print("\n=== BEFORE MOVE ===")
joints = iscoin.robot_control.get_actual_joint_positions()
print(f"Current joints: {joints}")
tcp = iscoin.robot_control.get_actual_tcp_pose()
print(f"Current TCP:    {tcp}")

home = tcp


=== BEFORE MOVE ===
Current joints: Joint6D([1.194510077457171, -1.1268384369341498, 1.0484009492530377, -1.5987887022488514, -1.5213804636174315, 1.0468793425442566])
Current TCP:    TCP6D([-0.02451002198137138, -0.4445511818512695, 0.20628591087811549, 1.993447266574015, 2.3179832815882655, -0.1289164020492196])


In [5]:
def ik_preflight(poses):
    """Return (ok, failed_index). failed_index is 1-based."""
    qnear = iscoin.robot_control.get_actual_joint_positions()
    for idx, pose in enumerate(poses, start=1):
        q = iscoin.robot_control.get_inverse_kin(pose, qnear=qnear)
        if q is None:
            return False, idx
        qnear = q
    return True, None


def get_pen(pen_pos):
    v = 0.2

    # Build a stable "home" pose with the same XYZ but fixed downward tool orientation
    current_tcp = iscoin.robot_control.get_actual_tcp_pose()

    print(f"current tcp: ({current_tcp.x}, {current_tcp.y}, {current_tcp.z}, {current_tcp.rx}, {current_tcp.ry}, {current_tcp.rz})")

    home_down = TCP6D.createFromMetersRadians(
        current_tcp.x,
        current_tcp.y,
        current_tcp.z,
        DOWN_RX,
        DOWN_RY,
        DOWN_RZ,
    )

    to_top_of_pen = TCP6D.createFromMetersRadians(
        pen_pos[0],
        pen_pos[1],
        pen_pos[2] + PEN_OFFSET,
        DOWN_RX,
        DOWN_RY,
        DOWN_RZ,
    )

    to_pen = TCP6D.createFromMetersRadians(
        pen_pos[0],
        pen_pos[1],
        pen_pos[2],
        DOWN_RX,
        DOWN_RY,
        DOWN_RZ,
    )

    planned_poses = [home_down, to_top_of_pen, to_pen, to_top_of_pen, home_down]
    ok_ik, failed_idx = ik_preflight(planned_poses)
    if not ok_ik:
        print(f"IK preflight failed at planned pose #{failed_idx}")
        return False

    print("\n=== MOVEL: going over pen position ===")
    waypoints_to_pen = [
        TCP6DDescriptor(home_down, v=v, r=0.01),
        TCP6DDescriptor(to_top_of_pen, v=v, r=0.01),
        TCP6DDescriptor(to_pen, v=v),
    ]

    waypoints_home = [
        TCP6DDescriptor(to_top_of_pen, v=v, r=0.01),
        TCP6DDescriptor(home_down, v=v),
    ]

    ok = iscoin.robot_control.movel_waypoints(waypoints_to_pen)
    if not ok:
        print("Pick approach failed")
        return False

    # iscoin.gripper.close()
    
    ok = iscoin.robot_control.movel_waypoints(waypoints_home)
    if not ok:
        print("Return-home motion failed")
        return False

    return True

In [6]:
# iscoin.gripper.activate()

In [7]:
get_pen(PEN_POS)

current tcp: (-0.024510021981285636, -0.4445511818510432, 0.20628591087760728, 1.9934472665745466, 2.3179832815887735, -0.12891640204658064)

=== MOVEL: going over pen position ===
movel_waypoints sent (3 points, total=6s)
Movement OK — target reached
movel_waypoints sent (1 points, total=2s)
Movement OK — target reached


True

In [8]:
# if reachable:
# if get_pen(PEN_POS):
#     print("Pick routine completed")
# else:
#     print("Pick routine failed (see IK preflight / motion message above)")
# else:
#     print("Position isn't reachable")

In [9]:
# iscoin.gripper.deactivate()